<a href="https://colab.research.google.com/github/Adhira-Deogade/pytorch-learnings/blob/main/cnn_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np
import torch.nn.functional as F
import skimage.io as io
from PIL import Image

First, mount your Google Drive. This will ask you to authenticate and grant Colab access to your Drive files.

In [2]:
from google.colab import drive
drive.mount('/content/drive')
# Replace 'my_document.txt' with the actual path to your file in Google Drive
file_path = '/content/drive/My Drive/Studies/Data/cnn.pth'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Once your Drive is mounted, you can access files by specifying the path, which will start with `/content/drive/My Drive/`. For example, if you have a file named `my_document.txt` in the root of your Drive, you can open it like this:

In [3]:
# Create the neural network
class ConvNet(nn.Module):
  def __init__(self):
     super(ConvNet, self).__init__()
     # ((W - F + 2P) / S) + 1 -> (32 - 3)/1 + 1 = 30
     self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3)
     self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
     # ((W - F + 2P) / S) + 1 -> (15 - 3)/1 + 1 = 13
     self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3)
     # ((W - F + 2P) / S) + 1 -> (6 - 3)/1 + 1 = 4
     self.conv3 = nn.Conv2d(in_channels=64, out_channels=64, kernel_size=3)
     # Flatten 3D tensor into 1D tensor before applying Linear layer, no pooling
     # Size of 1D tensor is 16*4*4
     self.fc1 = nn.Linear(in_features=64*4*4, out_features=64)
     #  10 classes
     self.fc2 = nn.Linear(in_features=64, out_features=10)

  def forward(self,x):
    # First layer: n, 3, 32, 32
    x = F.relu(self.pool(self.conv1(x))) # n, 32, 15, 15
    # Second layer
    x = F.relu(self.pool(self.conv2(x))) # n, 64, 6, 6
    # Third layer, no pooling
    x = F.relu(self.conv3(x)) # n, 64, 4, 4
    # Third layer - flatten the tensor from 3D to 2D
    x = x.view(-1, 64*4*4) # n, 1024
    x = F.relu(self.fc1(x)) # n, 1024, 64
    # No activation, because we are using CrossEntropyLoss which does that
    x = self.fc2(x) # n, 64, 10
    return x


In [4]:
convnet = ConvNet()
print(convnet)

ConvNet(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1))
  (conv3): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
  (fc1): Linear(in_features=1024, out_features=64, bias=True)
  (fc2): Linear(in_features=64, out_features=10, bias=True)
)


In [5]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [6]:
# Helper function to show what's in the dataset
def imshow(imgs):
    # imgs = imgs / 2 + 0.5   # unnormalize
    npimgs = imgs.numpy()
    plt.imshow(np.transpose(npimgs, (1, 2, 0)))
    plt.show()



In [7]:
def get_prediction_from_image_url(image_url=""):
  # Returns image_array to be fed to model
  # Complete pre-processing
  # First convert image_url to numpy array
  image_numpy = io.imread(image_url)
  print(f"image_numpy.shape = {image_numpy.shape}")

  # Show image
  # Image.fromarray(image_numpy)

  # Convert numpy array to PILImage
  # Resize PILImage to 32,32 to match model's inputs
  # Convert PILImage to torch tensor
  # Apply normalization to that tensor
  # define what transforms we want to apply

  # dataset has PILImages in range [0,1]
  # Normalize these tensor values between -1 and 1
  # For 3 channels, we provide mean and standard deviation of 0.5
  transform_to_square_tensor = transforms.Compose([
      transforms.ToPILImage(), # Convert numpy array to PIL Image
      transforms.Resize((32, 32)),
      transforms.ToTensor(), # This will convert to torch.float32 automatically
      transforms.Normalize((0.5,0.5,0.5), (0.5,0.5,0.5)) # Add normalization as per original plan
  ])
  image_tensor = transform_to_square_tensor(image_numpy)
  return image_tensor


In [8]:
def prediction_result(input_image=''):
  # Returns predicted index of the classes
  # from pre-processed image tensor
  with torch.no_grad():
    # Move tensor to GPU, that is what model uses
    image_tensor_gpu = input_image.to(device)

    # Feed image to NN
    output = convnet(image_tensor_gpu)

    # Get prediction class based on highest probability, by row
    _, predicted = torch.max(output, 1)
    predicted_class = class_from_index(predicted)

    return predicted_class

In [9]:
def model_preprocessing(model='', weights_path=''):

  # Load weights
  # Load state dictionary
  loaded_state_dict = torch.load(weights_path)

  # Load model's weights into architecture
  model.load_state_dict(loaded_state_dict)

  # Get shape
  # Print loaded model's state_dict
  print("Loaded model's state_dict:")
  for param_tensor in model.state_dict():
      print(param_tensor, "\t", model.state_dict()[param_tensor].size())


  # Move model to gpu
  model.to(device)

  # Use model for evaluation
  # Set model's internal configs to be better suited for evaluations
  model.eval()

  return model

In [10]:
def class_from_index(predicted_index=''):
  # These are the original labels
  # ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
  # We got 1 as the answer, which means automobile was correctly identified?
  # Create a dictionary with above data
  all_image_classes = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
  return all_image_classes[predicted_index]


In [11]:
# All image predictions URLs
# All images should have only 3 channels
# Get an external image
corvette_image_url = "https://github.com/andandandand/images-for-colab-notebooks/blob/main/1964-chevrolet-corvette-stingray.jpeg?raw=true"

aeroplane_url = "https://thumbs.dreamstime.com/b/aeroplane-13698602.jpg"

# Frog image
frog_image_url = "https://media.istockphoto.com/id/175397603/photo/frog.jpg?s=612x612&w=0&k=20&c=EMXlwg5SicJllr7gnSFUUjzwCGa1ciLjYD1bk8NvO2E="




In [12]:
# Model preprocessing
model = model_preprocessing(convnet, file_path)

Loaded model's state_dict:
conv1.weight 	 torch.Size([32, 3, 3, 3])
conv1.bias 	 torch.Size([32])
conv2.weight 	 torch.Size([64, 32, 3, 3])
conv2.bias 	 torch.Size([64])
conv3.weight 	 torch.Size([64, 64, 3, 3])
conv3.bias 	 torch.Size([64])
fc1.weight 	 torch.Size([64, 1024])
fc1.bias 	 torch.Size([64])
fc2.weight 	 torch.Size([10, 64])
fc2.bias 	 torch.Size([10])


In [13]:
# Get input pre-processed tensor
frog_tensor = get_prediction_from_image_url(frog_image_url)
print(frog_tensor.shape)

# Get prediction
print(prediction_result(frog_tensor))

image_numpy.shape = (408, 612, 3)
torch.Size([3, 32, 32])
frog


In [14]:
# Get input pre-processed tensor
corvette_tensor = get_prediction_from_image_url(corvette_image_url)
print(corvette_tensor.shape)

# Get prediction
print(prediction_result(corvette_tensor))

image_numpy.shape = (1282, 1920, 3)
torch.Size([3, 32, 32])
automobile


In [15]:
# Get input pre-processed tensor
aeroplane_tensor = get_prediction_from_image_url(aeroplane_url)
print(aeroplane_tensor.shape)

# Get prediction
print(prediction_result(aeroplane_tensor))

image_numpy.shape = (533, 800, 3)
torch.Size([3, 32, 32])
airplane


In [16]:
ship_url = "https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcScHAziBvaC8TAYegXJ48RT1FL1qCVUyMNw5Q&s"
# Get input pre-processed tensor
ship_tensor = get_prediction_from_image_url(ship_url)
print(ship_tensor.shape)

# Get prediction
print(prediction_result(ship_tensor))

image_numpy.shape = (183, 275, 3)
torch.Size([3, 32, 32])
ship


In [17]:
adhira_url = "https://img.freepik.com/premium-photo/container-truck-mockup-advertising-isolated-white-background_669798-7705.jpg?semt=ais_hybrid&w=740"
# Get input pre-processed tensor
adhira_tensor = get_prediction_from_image_url(adhira_url)
print(adhira_tensor.shape)

# Get prediction
print(prediction_result(adhira_tensor))

image_numpy.shape = (493, 740, 3)
torch.Size([3, 32, 32])
truck
